<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [1]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [ ]:
def vectorizar(texto, vocabulario):
    palabras = set(texto.lower().split())
    return [1 if w in palabras else 0 for w in vocabulario]

In [11]:
X = [vectorizar(texto, vocabulario) for _, texto in corpus]

In [12]:
for texto in ["gana dinero gratis", "reunion de proyecto"]:
    vec = vectorizar(texto, vocabulario)
    print(f"Texto: {texto!r}")
    for palabra, bit in zip(vocabulario, vec):
        print(f"  {palabra:9} -> {'aparece' if palabra in texto.split() else 'no aparece'} -> {bit}")
    print(f"  Vector: {vec}\n")

Texto: 'gana dinero gratis'
  dinero    -> aparece -> 1
  gratis    -> aparece -> 1
  premio    -> no aparece -> 0
  proyecto  -> no aparece -> 0
  reunion   -> no aparece -> 0
  Vector: [1, 1, 0, 0, 0]

Texto: 'reunion de proyecto'
  dinero    -> no aparece -> 0
  gratis    -> no aparece -> 0
  premio    -> no aparece -> 0
  proyecto  -> aparece -> 1
  reunion   -> aparece -> 1
  Vector: [0, 0, 0, 1, 1]



## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [13]:
etiquetas = [etiqueta for etiqueta, _ in corpus]

n_total  = len(corpus)
n_spam   = etiquetas.count("spam")
n_normal = etiquetas.count("normal")

print(f"Correos totales: {n_total}")
print(f"Correos spam   : {n_spam}")
print(f"Correos normal : {n_normal}")

Correos totales: 11
Correos spam   : 5
Correos normal : 6


In [15]:
P_spam   = n_spam / n_total
P_normal = n_normal / n_total

print(f"P(Y=spam)   = {n_spam}/{n_total} = {P_spam:.4f}")
print(f"P(Y=normal) = {n_normal}/{n_total} = {P_normal:.4f}")

P(Y=spam)   = 5/11 = 0.4545
P(Y=normal) = 6/11 = 0.5455


## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [16]:
import pandas as pd

conteo_spam   = [0] * len(vocabulario)
conteo_normal = [0] * len(vocabulario)

for (etiqueta, _), vec in zip(corpus, X):
    for i, bit in enumerate(vec):
        if etiqueta == "spam":
            conteo_spam[i] += bit
        else:
            conteo_normal[i] += bit

tabla_conteos = pd.DataFrame(
    {"spam": conteo_spam, "normal": conteo_normal},
    index=vocabulario
)
print(tabla_conteos)

          spam  normal
dinero       4       0
gratis       4       0
premio       2       1
proyecto     0       4
reunion      1       3


In [17]:
p_spam   = [c / n_spam   for c in conteo_spam]
p_normal = [c / n_normal for c in conteo_normal]

tabla_prob = pd.DataFrame(
    {"P(Xi=1 | spam)": p_spam, "P(Xi=1 | normal)": p_normal},
    index=vocabulario
)
print(tabla_prob.round(4))

          P(Xi=1 | spam)  P(Xi=1 | normal)
dinero               0.8            0.0000
gratis               0.8            0.0000
premio               0.4            0.1667
proyecto             0.0            0.6667
reunion              0.2            0.5000


Si una palabra nunca aparece en una clase, su frecuencia estimada es 0 para esa clase. Eso es un problema porque $P(X\mid Y)$ es un producto de factores, y un solo factor igual a 0 anula todo el producto

In [18]:
for i, palabra in enumerate(vocabulario):
    if conteo_spam[i] == 0:
        print(f"'{palabra}' nunca aparece en spam    -> P(Xi=1 | spam) = 0")
    if conteo_normal[i] == 0:
        print(f"'{palabra}' nunca aparece en normal  -> P(Xi=1 | normal) = 0")

'dinero' nunca aparece en normal  -> P(Xi=1 | normal) = 0
'gratis' nunca aparece en normal  -> P(Xi=1 | normal) = 0
'proyecto' nunca aparece en spam    -> P(Xi=1 | spam) = 0


## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [19]:
p_spam_lap   = [(c + 1) / (n_spam   + 2) for c in conteo_spam]
p_normal_lap = [(c + 1) / (n_normal + 2) for c in conteo_normal]

tabla_laplace = pd.DataFrame(
    {"P(Xi=1 | spam)": p_spam_lap, "P(Xi=1 | normal)": p_normal_lap},
    index=vocabulario
)
print(tabla_laplace.round(4))

          P(Xi=1 | spam)  P(Xi=1 | normal)
dinero            0.7143             0.125
gratis            0.7143             0.125
premio            0.4286             0.250
proyecto          0.1429             0.625
reunion           0.2857             0.500


In [20]:
comparacion = pd.DataFrame({
    "spam (sin)":    p_spam,
    "spam (Laplace)":   p_spam_lap,
    "normal (sin)":  p_normal,
    "normal (Laplace)": p_normal_lap,
}, index=vocabulario)
print(comparacion.round(4))

          spam (sin)  spam (Laplace)  normal (sin)  normal (Laplace)
dinero           0.8          0.7143        0.0000             0.125
gratis           0.8          0.7143        0.0000             0.125
premio           0.4          0.4286        0.1667             0.250
proyecto         0.0          0.1429        0.6667             0.625
reunion          0.2          0.2857        0.5000             0.500


## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [22]:
texto_nuevo = "dinero gratis"
x_nuevo = vectorizar(texto_nuevo, vocabulario)
print(f"{texto_nuevo!r} -> {x_nuevo}")

'dinero gratis' -> [1, 1, 0, 0, 0]


In [ ]:
def factores(x, probs_1):
    return [p if xi == 1 else 1 - p for xi, p in zip(x, probs_1)]

f_spam   = factores(x_nuevo, p_spam_lap)
f_normal = factores(x_nuevo, p_normal_lap)

tabla_factores = pd.DataFrame({
    "xi": x_nuevo,
    "P(Xi=xi | spam)": f_spam,
    "P(Xi=xi | normal)": f_normal,
}, index=vocabulario)
print(tabla_factores.round(4))

          xi  P(Xi=xi | spam)  P(Xi=xi | normal)
dinero     1           0.7143              0.125
gratis     1           0.7143              0.125
premio     0           0.5714              0.750
proyecto   0           0.8571              0.375
reunion    0           0.7143              0.500


In [28]:
import math

def clasificar(texto, vocabulario, p_spam_lap, p_normal_lap, P_spam, P_normal, verbose=True):
    x = vectorizar(texto, vocabulario)

    P_x_spam   = math.prod(factores(x, p_spam_lap))
    P_x_normal = math.prod(factores(x, p_normal_lap))

    S_spam   = P_spam   * P_x_spam
    S_normal = P_normal * P_x_normal

    suma        = S_spam + S_normal
    post_spam   = S_spam   / suma
    post_normal = S_normal / suma

    clase = "spam" if post_spam > post_normal else "normal"

    if verbose:
        print(f"Correo: {texto!r}  x = {x}")
        print(f"  P(x | spam)   = {P_x_spam:.6f}    P(x | normal) = {P_x_normal:.6f}")
        print(f"  S_spam        = {S_spam:.6f}    S_normal      = {S_normal:.6f}")
        print(f"  P(spam | x)   = {post_spam:.4f}      P(normal | x) = {post_normal:.4f}")
        print(f"  Clasificación: {clase.upper()}\n")

    return clase, post_spam, post_normal

In [29]:
clase, p_s, p_n = clasificar(
    "dinero gratis", vocabulario, p_spam_lap, p_normal_lap, P_spam, P_normal
)

Correo: 'dinero gratis'  x = [1, 1, 0, 0, 0]
  P(x | spam)   = 0.178497    P(x | normal) = 0.002197
  S_spam        = 0.081135    S_normal      = 0.001199
  P(spam | x)   = 0.9854      P(normal | x) = 0.0146
  Clasificación: SPAM



## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?
2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?
3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?


### 1. 

Se usa en el paso donde la verosimilitud conjunta se factoriza en un producto. Sin la hipótesis $X_i \perp X_j \mid Y$, no se podría escribir ese producto, y $P(x\mid Y)$ tendría que estimarse directamente como una distribución conjunta sobre los 5 bits.

### 2.

Porque la distribución conjunta de los 5 bits queda determinada por las 5 probabilidades marginales $P(X_i=1\mid Y)$ de cada clase.

- **Modelo completo:** $2^5-1=31$ parámetros por clase (62 en total, más el prior).
- **Naive Bayes:** 5 parámetros por clase (10 en total, más el prior).

Con $n$ palabras el modelo completo crece como $2^n$ y Naive Bayes como $n$. Además, con solo 11 correos sería imposible estimar bien 32 probabilidades por clase: la mayoría de los vectores no aparecerían nunca en el corpus. Con 5 parámetros por clase, cada uno se estima con todos los correos de su clase.

### 3.

Porque modela la distribución conjunta $P(X,Y)=P(Y)\,P(X\mid Y)$, es decir, describe **cómo se generan los correos**: primero se elige la clase con $P(Y)$ y luego, dado ese valor, cada palabra aparece o no con $P(X_i\mid Y)$. Con ese modelo se podrían incluso simular correos nuevos.

Un modelo **discriminativo** (como la regresión logística) estima directamente $P(Y\mid x)$ sin modelar cómo se generan los $x$.